In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [3]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = model.config.id2label
label2id = model.config.label2id

def _normalize_label(x):
    return str(x).strip().lower().replace("-", "_").replace(" ", "_")

norm_label2id = {_normalize_label(k): int(v) for k, v in label2id.items()}

if "entailment" in norm_label2id:
    entailment_id = norm_label2id["entailment"]
else:
    entailment_id = next(i for i, lab in id2label.items() if _normalize_label(lab) == "entailment")

if "contradiction" in norm_label2id:
    contradiction_id = norm_label2id["contradiction"]
else:
    contradiction_id = next(i for i, lab in id2label.items() if _normalize_label(lab) == "contradiction")

print(model_name)
print("id2label:", id2label)
print("entailment_id:", entailment_id, "contradiction_id:", contradiction_id)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli
id2label: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
entailment_id: 0 contradiction_id: 2


In [4]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
batch_size = 64
dir12_scores = []
dir21_scores = []
avg_scores = []
preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc12 = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc21 = tokenizer(
            batch_s2,
            batch_s1,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        enc12 = {k: v.to(device) for k, v in enc12.items()}
        enc21 = {k: v.to(device) for k, v in enc21.items()}

        logits12 = model(**enc12).logits
        logits21 = model(**enc21).logits

        log_probs12 = torch.log_softmax(logits12, dim=-1)
        log_probs21 = torch.log_softmax(logits21, dim=-1)

        score12 = (log_probs12[:, entailment_id] - log_probs12[:, contradiction_id]).detach().cpu().numpy()
        score21 = (log_probs21[:, entailment_id] - log_probs21[:, contradiction_id]).detach().cpu().numpy()
        score_avg = 0.5 * (score12 + score21)
        batch_preds = (score_avg > 0).astype(np.int64)

        dir12_scores.extend(score12.tolist())
        dir21_scores.extend(score21.tolist())
        avg_scores.extend(score_avg.tolist())
        preds.extend(batch_preds.tolist())

dir12_scores = np.array(dir12_scores)
dir21_scores = np.array(dir21_scores)
avg_scores = np.array(avg_scores)
y_pred = np.array(preds)

print("done")
print("pred_positive_rate:", float(y_pred.mean()))


  0%|          | 0/7 [00:00<?, ?it/s]

done
pred_positive_rate: 0.6225490196078431


In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.7034313725490197, 'f1': 0.7729831144465291}
                precision    recall  f1-score   support

not_paraphrase       0.53      0.63      0.57       129
    paraphrase       0.81      0.74      0.77       279

      accuracy                           0.70       408
     macro avg       0.67      0.68      0.67       408
  weighted avg       0.72      0.70      0.71       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("dir12_log_odds:", float(dir12_scores[i]))
    print("dir21_log_odds:", float(dir21_scores[i]))
    print("avg_log_odds:", float(avg_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
dir12_log_odds: 7.967843055725098
dir21_log_odds: 6.533724784851074
avg_log_odds: 7.250783920288086
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
dir12_log_odds: -9.776514053344727
dir21_log_odds: -0.8201398849487305
avg_log_odds: -5.2983269691467285
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
dir12_log_odds: -0.6154599189758301
dir

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("dir12_log_odds:", float(dir12_scores[i]))
    print("dir21_log_odds:", float(dir21_scores[i]))
    print("avg_log_odds:", float(avg_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


num_errors: 121
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
dir12_log_odds: -0.6154599189758301
dir21_log_odds: 6.547325134277344
avg_log_odds: 2.965932607650757
true: 0 pred: 1
idx: 4
sentence1: No dates have been set for the civil or the criminal trial .
sentence2: No dates have been set for the criminal or civil cases , but Shanley has pleaded not guilty .
dir12_log_odds: -1.2382020950317383
dir21_log_odds: 7.657938003540039
avg_log_odds: 3.2098679542541504
true: 0 pred: 1
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
dir12_log_odds: -1.3924

In [9]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
    "method": "mean_bidirectional_entailment_vs_contradiction_log_probability_odds",
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.7034313725490197,
 'f1': 0.7729831144465291,
 'method': 'mean_bidirectional_entailment_vs_contradiction_log_probability_odds'}